# Análisis exploratorio 

* Luisa Merlo García | A01067715
* Edgar Osvaldo Navarro | A01644488
* Ariana Guadalupe Rosales Villalobos | A01644773

## Cargamos el Dataset y revisamos sus dimensiones

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
import os
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats


# Ajusta la carpeta si el Excel no está junto al notebook
archivo = Path("Datos/Resumen Downtime Current.xlsx")

df_original = pd.read_excel(archivo, sheet_name="2024")

# Trabajamos con una copia para conservar los datos originales
df = df_original.copy()
df.columns = df.columns.str.strip()

print("Forma del dataset:", df.shape)
print("Cantidad de registros:", df.shape[0])
print("Cantidad de variables:", df.shape[1])

display(df.head())
df.info()

Forma del dataset: (10167, 28)
Cantidad de registros: 10167
Cantidad de variables: 28


,Equipo,Opened,Update,Reason,Ticket,Downtime,ISSUE 13,Description 9,Work Description (Close Details) 17,Dia,...,Año,Fecha completa,2nd class 29,ss,Corrective,Class2 29,Eq Type 26,Automotive? As on Aug-2024,Brand,Model
0,Preferida,2026-08-23 23:50:52.500,2026-08-24 00:14:31.837,MANTENIMIENTO - MANTTO. CORRECTIVO,REQ-1760551,0.39426,Mechanical Failure,Favor de checar giro intermedio de fin de braz...,Se extrae tonillo capado de sujecion de block ...,2026-08-23 23:50:52.500,...,2026,2026 - 8,Robot Wrist problem,1,Corrective,Robot Wrist problem,Robot,NO,SEPRO,S5-35 S3 AXIAL
1,LaBarranca,2026-08-23 23:25:43.213,2026-08-23 23:27:48.213,MANTENIMIENTO - MANTTO. CORRECTIVO,REQ-1760257,0.034722,Calibration Parameter Modification,Favor de revisar falla de presion de cierre,Restablece maquina 2 veces en turno\nSe ajusta...,2026-08-23 23:25:43.213,...,2026,2026 - 8,Clamping/ejector movement problems,1,Corrective,Clamping/ejector movement problems,IMM,NO,KRAUSS MAFFEI,KM420-1900C2
2,Ruisenores,2026-08-23 23:21:46.997,2026-08-24 00:05:28.297,MANTENIMIENTO - MANTTO. CORRECTIVO,REQ-1760550,0.728139,Clamping Piston Failure,Favor de checar nivel de aceite bajo se apaga ...,Realiza recuperacion de nivel de aceite en maq...,2026-08-23 23:21:46.997,...,2026,2026 - 8,Oil Alarm,1,Corrective,Oil Alarm,IMM,NO,NISSEI,FV9000
3,LaBarranca,2026-08-23 21:23:06.703,2026-08-23 21:28:53.060,MANTENIMIENTO - MANTTO. CORRECTIVO,REQ-1760257,0.09621,Calibration Parameter Modification,Favor de revisar falla de presion de cierre,Restablece maquina 2 veces en turno\nSe ajusta...,2026-08-23 21:23:06.703,...,2026,2026 - 8,Clamping/ejector movement problems,1,Corrective,Clamping/ejector movement problems,IMM,NO,KRAUSS MAFFEI,KM420-1900C2
4,Canada,2026-08-23 20:54:49.517,2026-08-23 21:05:09.210,MANTENIMIENTO - MANTTO. CORRECTIVO,REQ-1760499,0.172137,Process Adjustment,Favor de colocar barra donde se embobinado ro...,se volvio a fijar barra de el winder,2026-08-23 20:54:49.517,...,2026,2026 - 8,Foil feeder,1,Corrective,Foil feeder,Feeder,NO,WITTMANN,Feeder


<class 'pandas.DataFrame'>
RangeIndex: 10167 entries, 0 to 10166
Data columns (total 28 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0   Equipo                               10167 non-null  str           
 1   Opened                               10167 non-null  datetime64[us]
 2   Update                               10167 non-null  datetime64[us]
 3   Reason                               10167 non-null  str           
 4   Ticket                               10162 non-null  str           
 5   Downtime                             10167 non-null  object        
 6   ISSUE 13                             8383 non-null   object        
 7   Description 9                        8386 non-null   object        
 8   Work Description (Close Details) 17  8384 non-null   object        
 9   Dia                                  10167 non-null  datetime64[us]
 10  mes                  

El dataset contiene 10,167 registros y 28 variables. Cada fila representa un registro de downtime asociado a un equipo y, cuando está disponible, a un ticket. Un mismo ticket puede aparecer en varios registros, por lo que el número de filas no equivale al número de tickets únicos. Aunque la hoja se llama “2024”, las fechas de apertura abarcan del 2 de enero de 2025 al 23 de agosto de 2026.

#### Eliminamos los registros que en corrective sean no corrective error humano

In [30]:
# Opción: Eliminar "no corrective" (ignorando mayúsculas/minúsculas y espacios extra)
# pero manteniendo los valores nulos (NaN) en la columna:
# Ver conteo de valores únicos incluyendo los NaN
print(df["Corrective"].value_counts(dropna=False))

# Crear el filtro para excluir 'NOT CORRECTIVE' y valores anómalos como 'Failure in axis movements'
filtro = (
    df["Corrective"].notna()
    & (df["Corrective"].str.upper().str.strip() != "NOT CORRECTIVE")
    & (df["Corrective"].str.strip() != "Failure in axis movements")
)

# Aplicar el filtro al DataFrame y sobrescribirlo
df = df[filtro].copy()

# Verificar la nueva forma del dataset
print("Nueva forma del dataset:", df.shape)
print(df["Corrective"].value_counts(dropna=False))

Corrective
Corrective                   7716
NaN                          1258
NOT CORRECTIVE               1192
Failure in axis movements       1
Name: count, dtype: int64
Nueva forma del dataset: (7716, 28)
Corrective
Corrective    7716
Name: count, dtype: int64


#### Eliminamos columnas que tiene la misma informacion

In [31]:
# Buscar pares de columnas que tengan exactamente los mismos valores
columnas_identicas = []
cols = df.columns

for i in range(len(cols)):
  for j in range(i + 1, len(cols)):
    if df[cols[i]].equals(df[cols[j]]):
      columnas_identicas.append((cols[i], cols[j]))

print("Pares de columnas idénticas:")
for col1, col2 in columnas_identicas:
  print(f"- '{col1}' es idéntica a '{col2}'")

Pares de columnas idénticas:
- 'Opened' es idéntica a 'Dia'
- 'ISSUE 13' es idéntica a 'Clase 13'


In [32]:
# Reemplaza la lista con las columnas que quieras eliminar
columnas_a_borrar = [
    "Fecha completa",
    "2nd class 29",
]  # Ejemplo de columnas redundantes
df = df.drop(columns=columnas_a_borrar)

print("Nueva forma del dataset:", df.shape)

Nueva forma del dataset: (7716, 26)


In [45]:
# Convertimos la columna Downtime a numérica, forzando los errores a NaN
df['Downtime'] = pd.to_numeric(df['Downtime'], errors='coerce')

In [36]:
# Borramos la column 'Automotive? As on Aug-2024' por que tiene solo un valor
df.drop(columns=['Automotive? As on Aug-2024'], inplace=True, errors='ignore')
df = df.drop(columns=['Class2 29', 'Eq Type 26', 'EQ2'], errors='ignore')
df.drop(columns=['Corrective'], inplace=True, errors='ignore')
df.drop(columns=['Dia'], inplace=True, errors='ignore')
df.drop(columns=['Semana'], inplace=True, errors='ignore')

print("Nueva forma del dataset:", df.shape)

Nueva forma del dataset: (7716, 19)


In [37]:
df.info()

<class 'pandas.DataFrame'>
Index: 7716 entries, 0 to 10165
Data columns (total 19 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0   Equipo                               7716 non-null   str           
 1   Opened                               7716 non-null   datetime64[us]
 2   Update                               7716 non-null   datetime64[us]
 3   Reason                               7716 non-null   str           
 4   Ticket                               7716 non-null   str           
 5   Downtime                             7716 non-null   float64       
 6   ISSUE 13                             7207 non-null   object        
 7   Description 9                        7210 non-null   object        
 8   Work Description (Close Details) 17  7208 non-null   object        
 9   mes                                  7716 non-null   int64         
 10  Equipo 26                  

#### Duplicados

In [38]:
# Duplicados considerando todas las columnas
duplicados_completos = df.duplicated()

print("Filas completamente duplicadas:", duplicados_completos.sum())

# Posibles registros repetidos del mismo evento
clave_evento = ["Ticket", "Equipo", "Opened", "Update"]

print(
    "Repeticiones de la clave del evento:",
    df.duplicated(subset=clave_evento).sum()
)

# Revisamos la repetición de identificadores con formato REQ-número
tickets = df["Ticket"].astype("string").str.strip()
es_req = tickets.str.fullmatch(r"REQ-\d+", na=False)

frecuencia_tickets = tickets[es_req].value_counts()

print("\nRegistros con formato REQ-número:", es_req.sum())
print("Tickets REQ distintos:", tickets[es_req].nunique())
print(
    "Apariciones adicionales de tickets REQ:",
    tickets[es_req].duplicated().sum()
)

display(
    frecuencia_tickets[frecuencia_tickets > 1]
    .head(10)
    .rename("Cantidad de registros")
)

# Mostramos un ejemplo del ticket más frecuente
if not frecuencia_tickets.empty:
    ticket_ejemplo = frecuencia_tickets.index[0]

    display(
        df.loc[
            tickets.eq(ticket_ejemplo).fillna(False),
            ["Ticket", "Equipo", "Opened", "Update", "Downtime"]
        ].sort_values("Opened").head(10)
    )

Filas completamente duplicadas: 0
Repeticiones de la clave del evento: 0

Registros con formato REQ-número: 7206
Tickets REQ distintos: 4184
Apariciones adicionales de tickets REQ: 3022


Ticket
REQ-1684425    102
REQ-1518418     54
REQ-1682059     32
REQ-1678991     28
REQ-1602471     22
REQ-1742067     20
REQ-1760257     19
REQ-1730625     17
REQ-1603477     17
REQ-1614631     16
Name: Cantidad de registros, dtype: int64[pyarrow]

,Ticket,Equipo,Opened,Update,Downtime
2389,REQ-1684425,LaBarranca,2026-04-08 01:43:44.420,2026-04-08 01:47:27.543,0.061979
2388,REQ-1684425,LaBarranca,2026-04-08 02:13:47.460,2026-04-08 02:15:41.050,0.031553
2387,REQ-1684425,LaBarranca,2026-04-08 03:08:44.540,2026-04-08 03:10:06.147,0.022669
2386,REQ-1684425,LaBarranca,2026-04-08 04:13:32.713,2026-04-08 04:15:28.520,0.032169
2384,REQ-1684425,LaBarranca,2026-04-08 06:31:59.750,2026-04-08 06:33:47.760,0.030003
2380,REQ-1684425,LaBarranca,2026-04-08 11:21:34.440,2026-04-08 11:22:13.137,0.010749
2378,REQ-1684425,LaBarranca,2026-04-08 12:27:27.073,2026-04-08 12:28:25.107,0.016121
2376,REQ-1684425,LaBarranca,2026-04-08 13:47:04.073,2026-04-08 13:51:14.053,0.069439
2374,REQ-1684425,LaBarranca,2026-04-08 16:21:49.817,2026-04-08 16:25:20.510,0.058526
2372,REQ-1684425,LaBarranca,2026-04-08 16:55:07.113,2026-04-08 16:59:44.310,0.076999


In [39]:
import pandas as pd

# 1. Asegurar formato de fechas
df["Opened"] = pd.to_datetime(df["Opened"])
df["Update"] = pd.to_datetime(df["Update"])

# 2. Identificar cuáles tienen el formato REQ-
tickets = df["Ticket"].astype("string").str.strip()
es_req = tickets.str.fullmatch(r"REQ-\d+", na=False)

# Separar el dataset en dos
df_req = df[es_req].copy()
df_otros = df[~es_req].copy()

# 3. Definir reglas de agregación SOLO para los que tienen formato REQ-
reglas_agregacion = {
    col: "first"
    for col in df.columns
    if col not in ["Ticket", "Opened", "Update", "Downtime"]
}
reglas_agregacion["Opened"] = "min"
reglas_agregacion["Update"] = "max"

# Consolidar los tickets REQ
df_req_agrupado = df_req.groupby("Ticket", as_index=False).agg(
    reglas_agregacion
)

# Calcular el Downtime en horas para los tickets agrupados
df_req_agrupado["Downtime"] = (
    df_req_agrupado["Update"] - df_req_agrupado["Opened"]
).dt.total_seconds() / 3600

# 4. A los registros que NO son REQ, calcularles su Downtime individual
df_otros["Downtime"] = (
    df_otros["Update"] - df_otros["Opened"]
).dt.total_seconds() / 3600

# 5. Volver a unir ambos bloques en un solo DataFrame final
df_final = pd.concat([df_req_agrupado, df_otros], ignore_index=True)

print("Forma del dataset consolidado final:", df_final.shape)

Forma del dataset consolidado final: (4694, 19)


## Revisar valores faltantes

In [40]:
# Contamos valores faltantes de cada columna
faltantes = pd.DataFrame({
    "Cantidad": df.isna().sum(),
    "Porcentaje": (df.isna().mean() * 100).round(2)
})

faltantes = faltantes.sort_values("Cantidad", ascending=False)

display(faltantes)

print("Total de celdas faltantes:", df.isna().sum().sum())
print("Filas con al menos un faltante:", df.isna().any(axis=1).sum())

# También buscamos texto vacio o  espacios
columnas_texto = df.select_dtypes(include=["object", "string"]).columns

vacios = {}

for columna in columnas_texto:
    valores = df[columna].astype("string")
    vacios[columna] = valores.str.strip().eq("").sum()

display(
    pd.Series(vacios, name="Textos vacíos")
    .sort_values(ascending=False)
)

,Cantidad,Porcentaje
Model,629,8.15
Brand,537,6.96
ISSUE 13,509,6.60
Clase 13,509,6.60
Sub system 27,508,6.58
Work Description (Close Details) 17,508,6.58
Description 9,506,6.56
Equipo 26,67,0.87
Device 28,1,0.01
Equipo,0,0.00


Total de celdas faltantes: 3774
Filas con al menos un faltante: 1038


Equipo                                 0
Reason                                 0
Ticket                                 0
ISSUE 13                               0
Description 9                          0
Work Description (Close Details) 17    0
Equipo 26                              0
Sub system 27                          0
Clase 13                               0
DD                                     0
Device 28                              0
Brand                                  0
Model                                  0
Name: Textos vacíos, dtype: int64